# Example 01.10 — create an online model service

Creates one stable service endpoint with the promoted model as champion. A matching
running service is reused. An archived service keeps its governed name reserved and
produces a clear conflict instead of silently replacing history.

`MODEL_SERVICE_NAME` is the user-chosen service name used later for discovery and
online scoring.


In [1]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "ml_app_client").is_dir()),
    None,
)
if REPOSITORY_ROOT is None:
    raise RuntimeError("Start Jupyter inside the ml-app repository")
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from ml_app_client import MLAppClient, ResourceNotFoundError

client = MLAppClient.connect()
print("Connected to ML App")


Connected to ML App


## Choose resource names

These are normal names passed to `ml_app_client`. Edit them directly and use the
same values in the following notebooks. Dataset and pipeline names are resolved
inside the selected Business Case. Business Case and service names are globally
unique on one ML App installation.


In [2]:
# These are ordinary platform resource names. Change the two globally unique
# names (Business Case and model service) when sharing one installation.
BUSINESS_CASE_NAME = "[MLAPP EXAMPLE 01 v2] Estates Lifecycle - demo"
TRAINING_DATASET_NAME = "Example01 Estates - Training"
SCORING_DATASET_NAME = "Example01 Estates - Batch Input"
ACTUALS_DATASET_NAME = "Example01 Estates - Actuals"
TRAINING_PIPELINE_NAME = "Example01 03 - AutoML Training"
BATCH_PIPELINE_NAME = "Example01 05 - Batch Scoring"
MONITORING_PIPELINE_NAME = "Example01 07 - Performance Monitoring"
MODEL_NAME = "Example01 Estates Price Model"
OUTPUT_NAME_PREFIX = "Example01 Estates AutoML"
MODEL_SERVICE_NAME = "Example01 10 - Estates Model Service - demo"

TRAINING_RUN_KEY = "Example01-training-v2"
BATCH_RUN_KEY = "Example01-batch-scoring-v2"
MONITORING_RUN_KEY = "Example01-monitoring-v2"

print("Resource names configured for:", BUSINESS_CASE_NAME)


Resource names configured for: [MLAPP EXAMPLE 01 v2] Estates Lifecycle - demo


In [3]:
model = client.model_by_name(
    business_case_name=BUSINESS_CASE_NAME,
    model_name=MODEL_NAME,
)
if model["stage"] != "production":
    raise RuntimeError("Run Example01_09_promote_model.ipynb first")

try:
    service = client.deployment_by_name(MODEL_SERVICE_NAME, include_archived=True)
    created = False
    if service.status == "archived":
        raise RuntimeError("The example service is archived; choose a new scenario version")
    champion = next(item for item in service.active_revision["assignments"] if item["role"] == "champion")
    if str(champion["model_id"]) != str(model["id"]):
        raise RuntimeError("The existing example service uses a different champion model")
    if service.status == "stopped":
        service = client.set_deployment_status(service, status="running", reason="Resume Example01 service")
except ResourceNotFoundError:
    service = client.create_deployment(name=MODEL_SERVICE_NAME, model_id=str(model["id"]), retention_days=365)
    created = True
print({
    "action": "CREATED" if created else "FOUND",
    "service": service.name,
    "status": service.status,
    "endpoint": service.endpoint_url,
    "revision": service.active_revision.get("version_number"),
})


{'action': 'CREATED', 'service': 'Example01 10 - Estates Model Service - demo', 'status': 'running', 'endpoint': '/api/v1/serving/deployments/example01-10-estates-model-service-demo/predictions', 'revision': 1}
